In [ ]:
#2D
import netCDF4 as nc
import numpy as np

# Path to your NetCDF file
file_path = '<enter the path here>'
output_file_path = 'Converted.tec'

# Open the NetCDF file
nc_file = nc.Dataset(file_path, 'r')

# Extract node coordinates and face connectivity
node_x = nc_file.variables['mesh2d_node_x'][:]
node_y = nc_file.variables['mesh2d_node_y'][:]
face_nodes = nc_file.variables['mesh2d_face_nodes'][:]
node_z = nc_file.variables.get('mesh2d_node_z', np.zeros(len(node_x)))  # Handle missing variable gracefully

# Define the Tecplot ASCII file writer
with open(output_file_path, 'w') as f:
    # Header for Tecplot
    f.write("TITLE = \"Converted 2D Lake Mesh\"\n")
    f.write("VARIABLES = \"X\", \"Y\", \"Z\"\n")

    # Number of nodes and elements
    num_nodes = len(node_x)
    num_faces = len(face_nodes)
    f.write(f"ZONE N={num_nodes}, E={num_faces}, DATAPACKING=POINT, ZONETYPE=FETRIANGLE\n")

    # Write node data
    for i in range(num_nodes):
        z = node_z[i] if len(node_z) > i else 0.0  # Default Z to 0 if missing
        f.write(f"{node_x[i]} {node_y[i]} {z}\n")

    # Write connectivity data
    for face in face_nodes:
        f.write(f"{int(face[0])} {int(face[1])} {int(face[2])}\n")

# Close the NetCDF file
nc_file.close()

print(f"Tecplot file successfully created at: {output_file_path}")


In [ ]:
#Successfully merged the zones and applied SOLUTIONTIME and STRANDID as of this checkpoint 1.20.25
import netCDF4 as nc
import numpy as np

def calculate_surface_elevation_at_nodes(nc_file, time_step_index):
    """Calculate surface elevation at nodes by averaging over connected faces."""
    num_nodes = len(nc_file.variables['mesh2d_node_x'][:])
    num_faces = len(nc_file.variables['mesh2d_face_nodes'][:, 0])

    nodal_elevation_sum = np.zeros(num_nodes)
    nodal_counter = np.zeros(num_nodes)

    # face_nodes is 1-based in the NetCDF; shift to 0-based
    face_nodes = nc_file.variables['mesh2d_face_nodes'][:] - 1  
    # Use the specified time_step_index to select the correct frame of surface elevation
    surface_elevation_faces = nc_file.variables['mesh2d_s1'][time_step_index, :]

    for i in range(num_faces):
        nodes = face_nodes[i, :]
        for node in nodes:
            nodal_elevation_sum[node] += surface_elevation_faces[i]
            nodal_counter[node] += 1

    # Safely divide to handle any nodes not used in a face
    nodal_elevation = np.divide(
        nodal_elevation_sum, 
        nodal_counter,
        out=np.zeros_like(nodal_elevation_sum), where=nodal_counter != 0
    )
    return nodal_elevation

def generate_combined_layers(nc_file, time_indices, time_vals, 
                             num_sigma_layers, num_z_layers, 
                             interface_depth, layers_to_remove):
    """
    Generate combined sigma and Z layers for all time steps, ensuring:
      - Sigma layers go from fraction=0 (free surface) to fraction=1 (interface).
      - Z layers use one more 'surface' than requested volumetric layers.
    """

    node_x = nc_file.variables['mesh2d_node_x'][:]
    node_y = nc_file.variables['mesh2d_node_y'][:]
    # Convert to zero-based indexing
    face_nodes = nc_file.variables['mesh2d_face_nodes'][:] - 1  
    node_z = nc_file.variables['mesh2d_node_z'][:]

    # The full interface array in the NetCDF
    interface_z = nc_file.variables['mesh2d_interface_z'][:]
    # We'll slice out (num_z_layers+1) surfaces
    z_values = interface_z[layers_to_remove : layers_to_remove + (num_z_layers + 1)]
    
    all_layers = []

    for i, t_index in enumerate(time_indices):
        # Get the real time value in seconds (or other NetCDF time units).
        solution_time = time_vals[i]

        # 1) Calculate nodal surface elevation for this time step
        surface_elevation_nodes = calculate_surface_elevation_at_nodes(nc_file, t_index)

        # 2) Construct the vertical coordinates
        combined_points = []
        connectivity = []

        # ---- SIGMA SURFACES (0..num_sigma_layers => num_sigma_layers volumetric layers) ----
        sigma_points = []
        for sigma_i in range(num_sigma_layers + 1):
            # fraction goes from 0.0 to 1.0
            fraction = sigma_i / num_sigma_layers  
            # Linear interpolation from free surface down to interface_depth
            z_layer = surface_elevation_nodes + fraction * (interface_depth - surface_elevation_nodes)
            # Ensure sigma layers are above the lake bed
            z_layer = np.maximum(z_layer, node_z)

            for j in range(len(node_x)):
                sigma_points.append([node_x[j], node_y[j], z_layer[j]])

        # ---- Z SURFACES (we slice out num_z_layers+1 from interface_z) ----
        z_points = []
        for z_surface in z_values:
            # If z_surface is a single value, use np.maximum with node_z
            # because node_z is an array. We want the top to remain above bathymetry.
            adjusted_z = np.maximum(z_surface, node_z)
            for j in range(len(node_x)):
                z_points.append([node_x[j], node_y[j], adjusted_z[j]])

        # Combine everything into one big array of points
        # sigma_points first, then z_points
        combined_points = sigma_points + z_points

        # ======================
        # BUILD CONNECTIVITY
        # ======================

        # 1) SIGMA Connectivity
        #    We have (num_sigma_layers + 1) vertical surfaces => num_sigma_layers 3D layers
        #    So loop from 0..(num_sigma_layers - 1) for each face
        #    Actually we should loop 0..(num_sigma_layers - 1) or 0..(num_sigma_layers)?
        #    Because if you have 11 surfaces, that’s 10 volumetric layers => range(num_sigma_layers).
        for layer in range(num_sigma_layers):
            layer_offset = layer * len(node_x)
            next_layer_offset = (layer + 1) * len(node_x)
            for face in face_nodes:
                n1, n2, n3 = face
                # Tecplot uses 1-based node indexing
                connectivity.append([
                    n1 + layer_offset + 1, n2 + layer_offset + 1, n3 + layer_offset + 1, n3 + layer_offset + 1,
                    n1 + next_layer_offset + 1, n2 + next_layer_offset + 1, n3 + next_layer_offset + 1, n3 + next_layer_offset + 1
                ])

        # 2) Z Connectivity
        #    We have (num_z_layers + 1) vertical surfaces => num_z_layers volumetric layers
        total_sigma_nodes = (num_sigma_layers + 1) * len(node_x)
        for layer in range(num_z_layers):
            layer_offset = total_sigma_nodes + layer * len(node_x)
            next_layer_offset = total_sigma_nodes + (layer + 1) * len(node_x)
            for face in face_nodes:
                n1, n2, n3 = face
                connectivity.append([
                    n1 + layer_offset + 1, n2 + layer_offset + 1, n3 + layer_offset + 1, n3 + layer_offset + 1,
                    n1 + next_layer_offset + 1, n2 + next_layer_offset + 1, n3 + next_layer_offset + 1, n3 + next_layer_offset + 1
                ])

        # Keep track of everything for this time step
        all_layers.append((combined_points, connectivity, solution_time))

    return all_layers

def write_tecplot_file(output_file, layers):
    """Write the combined sigma and Z layers to a single Tecplot zone per time step."""
    with open(output_file, 'w') as f:
        f.write("TITLE = \"3D Lake Mesh with Combined Sigma and Z Layers (Single Zone per Time)\"\n")
        f.write("VARIABLES = \"X\", \"Y\", \"Z\"\n")

        # Each item in `layers` is (combined_points, connectivity, solution_time)
        for zone_idx, (combined_points, connectivity, sol_time) in enumerate(layers, start=1):
            # Typically, use the same StrandID if the geometry is consistent
            strand_id = 1

            f.write(
                f"ZONE T=\"Combined Layers Zone {zone_idx}\", "
                f"N={len(combined_points)}, E={len(connectivity)}, "
                f"F=FEBLOCK, ET=BRICK, "
                f"STRANDID={strand_id}, SOLUTIONTIME={sol_time:.2f}\n"
            )

            # Write coordinates in Tecplot block format
            # X-coords
            for point in combined_points:
                f.write(f"{point[0]:.8e}\n")
            # Y-coords
            for point in combined_points:
                f.write(f"{point[1]:.8e}\n")
            # Z-coords
            for point in combined_points:
                f.write(f"{point[2]:.8e}\n")

            # Write connectivity
            for face in connectivity:
                f.write(" ".join(map(str, face)) + "\n")

    print(f"TEC file '{output_file}' created successfully.")

def main():
    input_file = '<enter the path here>'
    output_file = 'Output.tec'

    # Open NetCDF file
    nc_file = nc.Dataset(input_file, 'r')

    # Indices of the time steps we wish to process (modify as needed)
    time_indices = [0, 1]  

    # Read the actual time values from the NetCDF
    all_times = nc_file.variables['time'][:]
    time_vals = all_times[time_indices]

    # Example user-defined parameters
    num_sigma_layers = 10     # e.g., 10 volumetric sigma layers => 11 surfaces
    num_z_layers = 50         # e.g., 50 volumetric Z-layers => 51 surfaces
    interface_depth = -5.03
    layers_to_remove = 0

    # Generate layers
    layers = generate_combined_layers(nc_file, 
                                      time_indices, 
                                      time_vals, 
                                      num_sigma_layers, 
                                      num_z_layers, 
                                      interface_depth, 
                                      layers_to_remove)

    # Write the Tecplot file
    write_tecplot_file(output_file, layers)

    nc_file.close()

if __name__ == "__main__":
    main()



In [ ]:
#Use this function to know about the variables and their specifications 
import netCDF4 as nc

def inspect_nc_variables(filepath):
    """
    Opens the specified NetCDF file and prints out:
      - Variable name
      - Dimensions and shape
      - Data type
      - Common attributes such as long_name, units, standard_name, etc.
    """
    with nc.Dataset(filepath, mode='r') as ds:
        print(f"Opened NetCDF file: {filepath}")
        print("-" * 60)
        
        # Iterate through each variable in the dataset
        for var_name in ds.variables:
            var_obj = ds.variables[var_name]
            
            # Basic metadata
            dims = var_obj.dimensions
            shape = var_obj.shape
            dtype = var_obj.dtype

            print(f"Variable: {var_name}")
            print(f"  Dimensions: {dims}")
            print(f"  Shape: {shape}")
            print(f"  Data Type: {dtype}")
            
            # Check common attributes
            # (Add or remove attributes as needed for your specific files)
            if hasattr(var_obj, 'long_name'):
                print(f"  long_name: {var_obj.long_name}")
            if hasattr(var_obj, 'units'):
                print(f"  units: {var_obj.units}")
            if hasattr(var_obj, 'standard_name'):
                print(f"  standard_name: {var_obj.standard_name}")
            
            print("-" * 60)

def main():
    # Example usage:
    nc_file_path = '<enter the path here>'  # Update with your actual file path
    inspect_nc_variables(nc_file_path)

if __name__ == "__main__":
    main()


In [ ]:
#Present code with issues for visualization 
#!/usr/bin/env python
# ---------------------------------------------------------------------------
#  Build a 3-D Tecplot file with sigma+Z layers and extra variables
#  Updated: 2025-05-12  (handles masked values, correct dim order)
# ---------------------------------------------------------------------------
import netCDF4 as nc
import numpy as np
from collections import defaultdict

# ------------ USER PARAMETERS ------------------------------------------------
INPUT_FILE   = "FlowFM_map.nc"
OUTPUT_FILE  = "Lake_Mesh_3D_with_IM1_TEMP_IM1S1.tec"

TIME_INDICES = [0, 1]        # which time steps to export
NUM_SIGMA    = 10            # 10 volumetric sigma layers → 11 surfaces
NUM_Z        = 50            # 50 volumetric Z-layers     → 51 surfaces
INTERFACE_Z  = -5.03         # depth where sigma ↔ Z transition happens
LAYERS_DROP  = 0             # skip shallowest Z surfaces if desired

# names exactly as in the NetCDF
VAR3D = ["mesh2d_IM1", "mesh2d_tem1"]     # 3-D variables (time, face, layer)
VAR2D = ["mesh2d_IM1S1"]                  # 2-D variables (time, face)
# -----------------------------------------------------------------------------

MISSING = -9.9e19          # sentinel written for blanks / masked values


# --------------------------------------------------------------------------- #
#                       ---  HELPER FUNCTIONS  ---                            #
# --------------------------------------------------------------------------- #
def face_to_node(face_nodes, face_vals, num_nodes):
    """
    Average a face-based 1-D array to nodes, ignoring NaNs.
    """
    acc   = np.zeros(num_nodes, dtype=float)
    count = np.zeros(num_nodes, dtype=int)

    for f, nodes in enumerate(face_nodes):
        val = face_vals[f]
        if np.isnan(val):
            continue
        for n in nodes:
            acc[n]   += val
            count[n] += 1

    out = np.full(num_nodes, MISSING, dtype=float)
    good = count > 0
    out[good] = acc[good] / count[good]
    return out


def var3d_to_nodes(nc_file, varname, t_idx, face_nodes, num_nodes):
    """
    Convert a 3-D variable stored as (time, face, layer) to (layer, node).
    """
    # getdata() drops the mask; fill_value→np.nan so we can ignore
    v_face_layer = np.ma.filled(
        nc_file.variables[varname][t_idx, :, :],
        np.nan
    )                                   # shape (face , layer)
    v_layer_face = v_face_layer.T       # → (layer, face)

    n_layers = v_layer_face.shape[0]
    out = np.empty((n_layers, num_nodes), dtype=float)

    for k in range(n_layers):
        out[k, :] = face_to_node(face_nodes, v_layer_face[k, :], num_nodes)

    return out                           # (layer, node)


def var2d_to_nodes(nc_file, varname, t_idx, face_nodes, num_nodes):
    """
    Convert a 2-D variable stored as (time, face) to (node).
    """
    v_face = np.ma.filled(
        nc_file.variables[varname][t_idx, :],
        np.nan
    )
    return face_to_node(face_nodes, v_face, num_nodes)   # (node,)


def surface_elev_nodes(nc_file, t_idx, face_nodes, num_nodes):
    """
    Nodal surface elevation from mesh2d_s1 (time, face).
    """
    s_face = np.ma.filled(nc_file.variables["mesh2d_s1"][t_idx, :], np.nan)
    return face_to_node(face_nodes, s_face, num_nodes)


# --------------------------------------------------------------------------- #
#                   ---  MAIN MESH / VARIABLE ASSEMBLY  ---                   #
# --------------------------------------------------------------------------- #
def build_layers(ds, times_idx, times_val,
                 n_sigma, n_z, interf_z, drop_z,
                 var3d, var2d):
    """
    Return list  of (points, connectivity, sol_time, var_blocks_dict)
    points        : [(x,y,z), ...]  length = nodes_per_surface × total_surfaces
    connectivity  : [[8 ints] ...]  brick connectivity (1-based for Tecplot)
    var_blocks    : dict var→list with same length as points
    """

    nx = ds.variables["mesh2d_node_x"][:]
    ny = ds.variables["mesh2d_node_y"][:]
    nz = ds.variables["mesh2d_node_z"][:]
    num_nodes = len(nx)

    face_nodes = ds.variables["mesh2d_face_nodes"][:] - 1    # 0-based
    interface_z_full = ds.variables["mesh2d_interface_z"][:]
    z_surfaces = interface_z_full[drop_z: drop_z + n_z + 1]

    # ---- pre-compute nodal variables for every needed (time, var) ----------
    nod3d = defaultdict(list)      # var → [ time0 (array), time1 (array), ... ]
    nod2d = defaultdict(list)

    for v in var3d:
        for t in times_idx:
            nod3d[v].append(var3d_to_nodes(ds, v, t, face_nodes, num_nodes))

    for v in var2d:
        for t in times_idx:
            nod2d[v].append(var2d_to_nodes(ds, v, t, face_nodes, num_nodes))

    layers_out = []

    # -------------------- LOOP  OVER  TIME ----------------------------------
    for pos, t_idx in enumerate(times_idx):
        sol_time = times_val[pos]
        selev = surface_elev_nodes(ds, t_idx, face_nodes, num_nodes)

        points       = []
        connectivity = []
        vblocks      = {v: [] for v in (var3d + var2d)}

        # ------------------ SIGMA SURFACES ----------------------------------
        n_model_layers = nod3d[var3d[0]][pos].shape[0]   # 60 in your file

        for s in range(n_sigma + 1):
            frac = s / n_sigma
            z_lay = selev + frac * (interf_z - selev)
            z_lay = np.maximum(z_lay, nz)

            # pick closest model layer for variable sampling
            lay_idx = int(round(frac * (n_model_layers - 1)))

            for n in range(num_nodes):
                points.append([nx[n], ny[n], z_lay[n]])

                for v in var3d:
                    vblocks[v].append(nod3d[v][pos][lay_idx, n])
                for v in var2d:
                    vblocks[v].append(nod2d[v][pos][n])

        # ------------------ Z SURFACES --------------------------------------
        for z_s in z_surfaces:
            z_adj = np.maximum(z_s, nz)
            for n in range(num_nodes):
                points.append([nx[n], ny[n], z_adj[n]])

                # use deepest model layer for Z part
                for v in var3d:
                    vblocks[v].append(nod3d[v][pos][-1, n])
                for v in var2d:
                    vblocks[v].append(nod2d[v][pos][n])

        # ------------------ CONNECTIVITY  (as before) -----------------------
        for lay in range(n_sigma):
            off_lo = lay * num_nodes
            off_hi = (lay + 1) * num_nodes
            for f in face_nodes:
                n1, n2, n3 = f
                connectivity.append([
                    n1+off_lo+1, n2+off_lo+1, n3+off_lo+1, n3+off_lo+1,
                    n1+off_hi+1, n2+off_hi+1, n3+off_hi+1, n3+off_hi+1
                ])

        tot_sigma_pts = (n_sigma + 1) * num_nodes
        for lay in range(n_z):
            off_lo = tot_sigma_pts + lay * num_nodes
            off_hi = tot_sigma_pts + (lay + 1) * num_nodes
            for f in face_nodes:
                n1, n2, n3 = f
                connectivity.append([
                    n1+off_lo+1, n2+off_lo+1, n3+off_lo+1, n3+off_lo+1,
                    n1+off_hi+1, n2+off_hi+1, n3+off_hi+1, n3+off_hi+1
                ])

        layers_out.append((points, connectivity, sol_time, vblocks))

    return layers_out


# --------------------------------------------------------------------------- #
#                       ---  TEC PLOT WRITER  ---                             #
# --------------------------------------------------------------------------- #
def write_tecplot(fname, layers, var_order):
    with open(fname, "w") as f:
        hdr = "\"X\" \"Y\" \"Z\" " + " ".join(f"\"{v}\"" for v in var_order)
        f.write("TITLE = \"3D Lake Mesh – sigma+Z with IM1, TEMP, IM1S1\"\n")
        f.write(f"VARIABLES = {hdr}\n")

        for idx, (pts, conn, sol_t, vblk) in enumerate(layers, start=1):
            f.write(
                f"ZONE T=\"LayerCombo {idx}\", "
                f"N={len(pts)}, E={len(conn)}, "
                f"F=FEBLOCK, ET=BRICK, "
                f"STRANDID=1, SOLUTIONTIME={sol_t:.6f}\n"
            )

            # coordinates
            for coord in range(3):
                for p in pts:
                    val = p[coord]
                    f.write(f"{val:.8e}\n")

            # extra variables
            for v in var_order:
                for val in vblk[v]:
                    f.write(f"{val if np.isfinite(val) else MISSING:.8e}\n")

            # connectivity
            for c in conn:
                f.write(" ".join(map(str, c)) + "\n")

    print(f"[OK] Tecplot file written: {fname}")


# --------------------------------------------------------------------------- #
#                                    MAIN                                     #
# --------------------------------------------------------------------------- #
def main():
    ds = nc.Dataset(INPUT_FILE, "r")
    time_vals = ds.variables["time"][TIME_INDICES]

    layers = build_layers(
        ds, TIME_INDICES, time_vals,
        NUM_SIGMA, NUM_Z, INTERFACE_Z, LAYERS_DROP,
        VAR3D, VAR2D
    )

    write_tecplot(OUTPUT_FILE, layers, VAR3D + VAR2D)
    ds.close()


if __name__ == "__main__":
    main()
